In [ ]:
#1)	Methode 1: Tussen batches : voor elke conditie (eg T0): neem median van de hele plaat eg plaat 1 , en doe dan features weg met het grootste verschil tussen platen van die conditie

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading data...")
df = pd.read_csv(INPUT_CSV)

# Identify columns
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. PRE-CLEANING (CRITICAL)
# ==========================================
# We must remove features that are constant (zero variance) across the WHOLE dataset.
# Otherwise, they will have 0 variance between plates and be ranked as "best".
overall_std = df[feature_cols].std()
active_features = overall_std[overall_std > 1e-6].index.tolist()
print(f"Dropped {len(feature_cols) - len(active_features)} invariant features.")

# ==========================================
# 2. CONDITION-BASED STABILITY FILTER
# ==========================================
print("Calculating inter-plate stability per condition...")

# Extract Condition (e.g., 'T0') from Plate name (e.g., 'PLATE1_T0')
df['Condition'] = df['Plate'].apply(lambda x: x.split('_')[-1])

# A. Calculate the Median for every feature per Plate
plate_medians = df.groupby(['Condition', 'Plate'])[active_features].median()

# B. Calculate Variance across plates within each Condition
# This tells us: "How much does this feature drift between Plate A and Plate B of T0?"
inter_plate_variance = plate_medians.groupby('Condition').var()

# C. Aggregate: Take the mean variance across all conditions (T0, T1, T2)
mean_volatility = inter_plate_variance.mean(axis=0)

# D. Selection: Keep the 200 features with the LOWEST inter-plate variance
top_n = 200
stable_features = mean_volatility.sort_values(ascending=True).head(top_n).index.tolist()

# ==========================================
# 3. FINAL EXPORT
# ==========================================
df_final = df[metadata_cols + stable_features]
output_path = os.path.join(OUTPUT_DIR, "featuresSelection_medianfullplates.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Features selected: {len(stable_features)}")
print(f"Top 3 most stable features: {stable_features[:3]}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","featuresSelection_medianfullplates.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_Method1")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#3)	Methode 3: per plaat normalizeer tov nosgrna : - median/sigma zodat alle nosgrnas van alle platen (mediaangewijs op 0 komen te liggen): plot wel enkel T0 en T1 zo, (T2 kan je erbij proberen maar ni echt de bedoeling

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

print("Loading data...")
df = pd.read_csv(INPUT_CSV)

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. WITHIN-PLATE CONTROL NORMALIZATION
# ==========================================
print("Normalizing features based on within-plate controls...")

def normalize_by_control(plate_df):
    # Filter for controls within this specific plate
    ctrl_mask = plate_df['Treatment'] == CONTROL_LABEL
    ctrls = plate_df.loc[ctrl_mask, feature_cols]
    
    # Calculate stats for this plate's controls
    ctrl_median = ctrls.median()
    ctrl_std = ctrls.std()
    
    # Handle edge case: if std is 0 (invariant feature), replace with 1 to avoid div by zero
    ctrl_std = ctrl_std.replace(0, 1)
    
    # Apply transformation: (x - median) / std
    # We apply this to the whole plate_df (all treatments)
    plate_df[feature_cols] = (plate_df[feature_cols] - ctrl_median) / ctrl_std
    return plate_df

# Group by Plate and apply the normalization function
df_norm = df.groupby('Plate', group_keys=False).apply(normalize_by_control)

# ==========================================
# 2. FINAL SAVE
# ==========================================
output_path = os.path.join(OUTPUT_DIR, "z_scored_to_controls.csv")
df_norm.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Normalized {len(df_norm)} wells across {df['Plate'].nunique()} plates.")
print(f"Control Median for feature '0' after normalization: {df_norm[df_norm['Treatment']==CONTROL_LABEL]['0'].median():.4f}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","z_scored_to_controls.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_zscored")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#2)	Methode 2: Median scalen: neem median van contoles over hele conditie, zorg dan dat median van de plaat daarop komt 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

print("Loading data...")
df = pd.read_csv(INPUT_CSV)
df['Condition'] = df['Plate'].apply(lambda x: x.split('_')[-1])

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. MEDIAN SCALING (SHIFTING)
# ==========================================
print("Applying Median Scaling to align Plates with Control Medians...")

# Step A: Calculate the target median (The "Goal") for each feature per Condition
# This tells us where the 'no_sgRNA' naturally sits for T0, T1, T2.
condition_control_medians = df[df['Treatment'] == CONTROL_LABEL].groupby('Condition')[feature_cols].median()

def align_plate_to_control(plate_df):
    condition = plate_df['Condition'].iloc[0]
    
    # The target baseline for this condition
    target_median = condition_control_medians.loc[condition]
    
    # The current center of THIS specific plate (using all wells)
    current_plate_median = plate_df[feature_cols].median()
    
    # Calculate the shift needed to move the plate median to the control median
    # Shift = Target - Current
    shift = target_median - current_plate_median
    
    # Apply the shift to all features in the plate
    plate_df[feature_cols] = plate_df[feature_cols] + shift
    return plate_df

# Apply the alignment per Plate
df_scaled = df.groupby('Plate', group_keys=False).apply(align_plate_to_control)

# ==========================================
# 2. FINAL SAVE
# ==========================================
output_path = os.path.join(OUTPUT_DIR, "median_scaled_to_controls.csv")
df_scaled.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Aligned {df['Plate'].nunique()} plates to their respective {df['Condition'].unique()} control baselines.")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","median_scaled_to_controls.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_MethodMedian scaling")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","median_scaled_to_controls.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_MethodMedian scalingT1enkel")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T1","PLATE2_T1",
                   "PLATE3_T1","PLATE4_T1",
                   "PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","aggregated_wells_median_min5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_T1enkel_geenpreprocess")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T1","PLATE2_T1",
                   "PLATE3_T1","PLATE4_T1",
                   "PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#na scaling median per conditoe, nu ook tussen conidtie

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "20marchecht", "median_scaled_to_controls.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

df = pd.read_csv(INPUT_CSV)
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. TARGETED CROSS-CONDITION ALIGNMENT (T0 & T1 ONLY)
# ==========================================
print("Aligning T0 and T1 to a shared baseline (Excluding T2)...")

# Step A: Define the reference groups
target_conditions = ['T0', 'T1']

# Step B: Calculate the Grand Median using ONLY T0 and T1 controls
ref_ctrls = df[(df['Treatment'] == CONTROL_LABEL) & (df['Condition'].isin(target_conditions))]
grand_median_T0_T1 = ref_ctrls[feature_cols].median()

def align_early_timepoints(cond_df):
    condition = cond_df['Condition'].iloc[0]
    
    # If the condition is T0 or T1, we shift it to the shared baseline
    if condition in target_conditions:
        current_ctrl_median = cond_df[cond_df['Treatment'] == CONTROL_LABEL][feature_cols].median()
        shift = grand_median_T0_T1 - current_ctrl_median
        cond_df[feature_cols] = cond_df[feature_cols] + shift
        print(f" -> Aligned {condition} to shared T0-T1 baseline.")
    else:
        # T2 (and any others) remain untouched
        print(f" -> Skipped {condition} (preserved original state).")
        
    return cond_df

# Apply the alignment per Condition
df_final_aligned = df.groupby('Condition', group_keys=False).apply(align_early_timepoints)

# ==========================================
# 2. SAVE
# ==========================================
output_path = os.path.join(OUTPUT_DIR, "medianconditionandtime.csv")
df_final_aligned.to_csv(output_path, index=False)

print("\n" + "="*40)
print("WORKFLOW COMPLETE")
print(f"T0 and T1 are now comparable. T2 remains independent.")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","medianconditionandtime.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_mediantussenconidites")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#standardization after median scaling

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "20marchecht", "medianconditionandtime.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

df = pd.read_csv(INPUT_CSV)
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. T0-T1 TARGETED STANDARDIZATION
# ==========================================
print("Standardizing T0 and T1 features based on shared control variance...")

# A. Isolate the reference group (T0 and T1 Controls)
t0_t1_mask = df['Condition'].isin(['T0', 'T1'])
ref_ctrls = df[t0_t1_mask & (df['Treatment'] == CONTROL_LABEL)]

# B. Calculate the Standard Deviation of these controls
# We use a small 'epsilon' (1e-6) to prevent division by zero for invariant features
ctrl_std = ref_ctrls[feature_cols].std().replace(0, 1) 

# C. Apply the Scaling ONLY to T0 and T1 rows
# Note: We don't subtract the median here because your previous 
# 'Median Scaling' step already moved their center to ~0.
df.loc[t0_t1_mask, feature_cols] = df.loc[t0_t1_mask, feature_cols] / ctrl_std

print(f" -> T0 and T1 features scaled by T0/T1 control variance.")
print(f" -> T2 features left in original median-scaled state.")

# ==========================================
# 2. SAVE FOR UMAP
# ==========================================
output_path = os.path.join(OUTPUT_DIR, "final_scaled_T0T1_for_umap.csv")
df.to_csv(output_path, index=False)

print("\n" + "="*40)
print("WORKFLOW COMPLETE")
print("Controls for T0 and T1 should now cluster tightly in UMAP.")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#precies geen verschil met vorige

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","final_scaled_T0T1_for_umap.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_medianANDstandardized")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#sphering after median scaling

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP & LOADING
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw aggregated data...")
df = pd.read_csv(INPUT_CSV)
df['Condition'] = df['Plate'].apply(lambda x: x.split('_')[-1])

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# --- STEP 1: VARIANCE FILTER (Safety) ---
# Remove features that have no variation at all to prevent math errors later
initial_count = len(feature_cols)
feat_std = df[feature_cols].std()
feature_cols = feat_std[feat_std > 1e-6].index.tolist()
print(f"Removed {initial_count - len(feature_cols)} invariant features.")

# ==========================================
# 2. WITHIN-PLATE MEDIAN SCALING
# ==========================================
print("Step 2: Aligning each plate to its own control median...")

def scale_plate(plate_df):
    # Target: The median of the controls on THIS specific plate
    ctrl_median = plate_df[plate_df['Treatment'] == CONTROL_LABEL][feature_cols].median()
    # Shift the whole plate so its center sits on its control median
    plate_df[feature_cols] = plate_df[feature_cols] - ctrl_median
    return plate_df

df_step2 = df.groupby('Plate', group_keys=False).apply(scale_plate)

# ==========================================
# 3. TARGETED CONDITION ALIGNMENT (T0 & T1)
# ==========================================
print("Step 3: Bringing T0 and T1 together (Targeted Alignment)...")

target_conds = ['T0', 'T1']
# Calculate a shared baseline for T0 and T1 controls
ref_ctrls = df_step2[(df_step2['Treatment'] == CONTROL_LABEL) & (df_step2['Condition'].isin(target_conds))]
grand_median_early = ref_ctrls[feature_cols].median()

def align_early(cond_df):
    cond = cond_df['Condition'].iloc[0]
    if cond in target_conds:
        current_ctrl_median = cond_df[cond_df['Treatment'] == CONTROL_LABEL][feature_cols].median()
        shift = grand_median_early - current_ctrl_median
        cond_df[feature_cols] = cond_df[feature_cols] + shift
    return cond_df

df_step3 = df_step2.groupby('Condition', group_keys=False).apply(align_early)
# This output is your 'targeted_aligned_features'

# ==========================================
# 4. FINAL STANDARDIZATION (Scaling the Spread)
# ==========================================
print("Step 4: Standardizing T0/T1 spread to collapse UMAP clusters...")

# Calculate standard deviation of T0/T1 controls
t0_t1_mask = df_step3['Condition'].isin(target_conds)
ref_ctrls_final = df_step3[t0_t1_mask & (df_step3['Treatment'] == CONTROL_LABEL)]
ctrl_std = ref_ctrls_final[feature_cols].std().replace(0, 1)

# Divide T0/T1 features by this standard deviation
df_step3.loc[t0_t1_mask, feature_cols] = df_step3.loc[t0_t1_mask, feature_cols] / ctrl_std

# ==========================================
# 5. EXPORT
# ==========================================
final_output_path = os.path.join(OUTPUT_DIR, "final_preprocessed_ready_for_umap.csv")
df_step3.to_csv(final_output_path, index=False)

print("\n" + "="*40)
print("PREPROCESSING COMPLETE")
print(f"Final file: {final_output_path}")
print("========================================")

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","final_preprocessed_ready_for_umap.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_medianANDstandardized")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#nog een pogin featuer selection after median scaling

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP & LOAD ALIGNED DATA
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "20marchecht", "medianconditionandtime.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

df = pd.read_csv(INPUT_CSV)
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. NOISE-BASED SELECTION (T0 & T1 CONTROLS ONLY)
# ==========================================
# CRITICAL: We exclude T2 from the stability calculation logic
df_ref = df[(df['Treatment'] == CONTROL_LABEL) & (df['Condition'].isin(['T0', 'T1']))].copy()

print(f"Calculating stability based on T0 and T1 controls ({len(df_ref)} wells)...")

# A. WITHIN-PLATE STABILITY (T0 & T1)
# Calculate noise within plates for early timepoints
print(" -> Step 1: Evaluating Within-Plate Consistency (T0/T1)...")
within_plate_std = df_ref.groupby('Plate')[feature_cols].std().mean()

# Keep features in the lowest 60% of noise
thresh_within = within_plate_std.quantile(0.60)
consistent_features = within_plate_std[within_plate_std < thresh_within].index.tolist()

# B. BETWEEN-PLATE STABILITY (T0 & T1)
# Calculate how much the control medians shift between T0 and T1 plates
print(" -> Step 2: Evaluating Between-Plate Stability (T0/T1)...")
plate_medians = df_ref.groupby('Plate')[consistent_features].median()
between_plate_std = plate_medians.std()

# Keep features in the lowest 50% of batch-variance
thresh_between = between_plate_std.quantile(0.50)
stable_features = between_plate_std[between_plate_std < thresh_between].index.tolist()

# ==========================================
# 2. REDUNDANCY REMOVAL (CORRELATION)
# ==========================================
# Use the whole dataset (or just T0/T1) to find redundant features
print(" -> Step 3: Removing Redundant Features (r > 0.9)...")
corr_matrix = df[stable_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

final_feature_list = [f for f in stable_features if f not in to_drop]

# ==========================================
# 3. SAVE VETTED DATASET (INCLUDING T2)
# ==========================================
# We apply the discovered "good" features to the entire dataframe
df_final = df[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vetted_aligned_no_T2_selection.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print("FEATURE SELECTION SUMMARY (T0/T1 Basis)")
print(f"Starting Features: {len(feature_cols)}")
print(f"Final Count (Non-Redundant): {len(final_feature_list)}")
print(f"Condition T2 rows preserved with these features: {len(df_final[df_final['Condition']=='T2'])}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","vetted_aligned_no_T2_selection.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_medianANDstandardized")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP & LOAD ALIGNED DATA
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "20marchecht", "medianconditionandtime.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "20marchecht")
CONTROL_LABEL = "no_sgRNA"

# TARGET SETTINGS
TARGET_FEATURE_COUNT = 10  # Change this to your desired final number

df = pd.read_csv(INPUT_CSV)
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Condition']
feature_cols = [c for c in df.columns if c not in metadata_cols]

# ==========================================
# 1. PRE-FLIGHT: REMOVE INVARIANT (T0/T1 BASIS)
# ==========================================
print("Step 0: Identifying Invariant Features in T0/T1...")
df_early = df[df['Condition'].isin(['T0', 'T1'])]
feat_std_early = df_early[feature_cols].std()

# Features must have some movement to be considered
active_features = feat_std_early[feat_std_early > 1e-6].index.tolist()
print(f" -> Removed {len(feature_cols) - len(active_features)} constant features.")

# ==========================================
# 2. NOISE-BASED SELECTION (T0 & T1 CONTROLS ONLY)
# ==========================================
df_ref = df_early[df_early['Treatment'] == CONTROL_LABEL].copy()

# A. WITHIN-PLATE STABILITY
print(f" -> Step 1: Ranking by Within-Plate Consistency (T0/T1)...")
within_plate_std = df_ref.groupby('Plate')[active_features].std().mean()

# To reach exactly 200 at the end, we start with a slightly larger pool 
# (e.g., 500) so that Step 2 and 3 have room to filter.
top_within = within_plate_std.sort_values(ascending=True).head(500).index.tolist()

# B. BETWEEN-PLATE STABILITY
print(f" -> Step 2: Ranking by Between-Plate Stability (T0/T1)...")
plate_medians = df_ref.groupby('Plate')[top_within].median()
between_plate_std = plate_medians.std()

# We take the top performers here before the correlation check
stable_features = between_plate_std.sort_values(ascending=True).head(210).index.tolist()

# ==========================================
# 3. REDUNDANCY REMOVAL & FINAL COUNT
# ==========================================
print(f" -> Step 3: Removing Redundancy & Finalizing to Top {TARGET_FEATURE_COUNT}...")

# 1. Calculate correlations
corr_matrix = df[stable_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# 2. Drop highly redundant ones first (r > 0.9)
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]
non_redundant = [f for f in stable_features if f not in to_drop]

# 3. Final Cut: Take the best remaining features based on the 'Between-Plate' rank
final_feature_list = non_redundant[:TARGET_FEATURE_COUNT]

# ==========================================
# 4. SAVE VETTED DATASET
# ==========================================
df_final = df[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vetted_fixed_count.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print("FEATURE SELECTION SUMMARY")
print(f"Target Count:          {TARGET_FEATURE_COUNT}")
print(f"Actual Features Saved: {len(final_feature_list)}")
print(f"Top Feature Kept:      {final_feature_list[0]}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"20marchecht","vetted_fixed_count.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_medianANDstandardized4")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1",
                   "PLATE3_T0","PLATE3_T1","PLATE4_T0","PLATE4_T1",
                   "PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
#only slecito nbased on T1, de oude selciotn

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "10marchecht", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "10marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=3000):
    """Ranks features by how similar the controls are within each plate."""
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Calculate std for each plate, then average those stds across plates
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability_t1(df, features, top_n_to_keep=200):
    """Ranks features by how stable the medians are across different T1 plates."""
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    
    if len(plate_medians) < 2:
        print("Warning: Only one T1 plate found. Skipping across-plate stability filter.")
        return features[:top_n_to_keep]

    # Calculate standard deviation of the medians across the T1 plates
    t1_batch_noise = plate_medians.std()
    stable_features = t1_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    """Removes highly correlated features to reduce redundancy."""
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE (T1 CONDITION ONLY)
# ==========================================

# Create a calibration subset using only plates that belong to condition T1
print("\nCreating T1-specific subset for feature selection...")
df_t1 = df_raw[df_raw['Plate'].str.contains('T1', case=False, na=False)].copy()

if df_t1.empty:
    raise ValueError("No T1 plates found. Please check your 'Plate' column naming.")

# --- STEP 0: VARIANCE FILTER (On T1) ---
print(f"Step 0: Variance Filter on T1 (std > 0.01)...")
t1_std = df_t1[feature_cols].std()
active_features = t1_std[t1_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY (On T1) ---
print(f"Step 1: Within-Plate Consistency on T1 (Top 3000)...")
step1_features = filter_within_plate_consistency(df_t1, active_features, top_n_to_keep=3000)

# --- STEP 2: ACROSS-PLATE STABILITY (On T1) ---
print(f"Step 2: Across-Plate Stability on T1 (Top 200)...")
step2_features = filter_across_plate_stability_t1(df_t1, step1_features, top_n_to_keep=200)

# --- STEP 3: REDUNDANCY REMOVAL (On T1) ---
print(f"Step 3: Redundancy Filter on T1 (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_t1, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE (Applying selection to ALL plates)
# ==========================================
# We take the features selected via T1 and keep them for the entire original dataframe
df_final = df_raw[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_10march_T1_selciton.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (T1 Condition Calibration)")
print(f"Original features: {len(feature_cols)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Rows preserved: {len(df_final)} (Across all T0, T1, T2 plates)")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"10marchecht","vettedcellcounts_10march_T1_selciton.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"20marchecht", "UMAP_medianANDstandardized4")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T1","PLATE2_T1","PLATE3_T1","PLATE4_T1","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()